<a href="https://colab.research.google.com/github/RuiRodrigues-lab/DataScienceFE/blob/Locker/CP2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
#Precisamos desta biblioteca para podermos escolher um ficheiro local
#Se o ficheiro vier por API ou tivermos um link, é so alterar a forma de import
from google.colab import files

# 1️⃣ Faz upload do ficheiro (vai abrir uma janela para escolher no teu PC)
uploaded = files.upload()

# 2️⃣ Guarda o nome do ficheiro (Colab mostra o nome depois do upload)
filename = list(uploaded.keys())[0]

# 3️⃣ Lê o Excel, por default lê sempre a primeira tab, por isso podemos usar o "sheet_name"
# Se tivermos dados em varias tabs, devemos usar uma Dataframe(df) para cada uma das tabs
df = pd.read_excel(filename, sheet_name='Dados')
df.head()

Saving CP2.xlsx to CP2 (7).xlsx


,Idade,Sexo,Residencia,Cohabitação,Deslocação,HorasEstudo,Altura,Peso,Cinema,SatisfaçãoLic
0,20,Masculino,Setúbal,Pais,10,3.0,178,83,Ficção Científica,Satisfeito
1,21,Feminino,Setúbal,Colegas,30,8.0,170,65,Comédia,Muito Satisfeito
2,20,Feminino,Setúbal,Colegas,15,6.0,165,55,Comédia,Satisfeito
3,21,Feminino,Setúbal,Colegas,45,12.0,170,55,Ficção Científica,Pouco Satisfeito
4,20,Feminino,Setúbal,Colegas,25,3.0,150,46,Comédia,Satisfeito


In [79]:
#A função df.shape devolve o numero de linhas e colunas do Dataframe(df)
linhas, colunas = df.shape
print("Número de linhas:", linhas)
print("Número de colunas:", colunas)

#Tambem temos a função df.describe que é muito gira mas os resultados podem ser um pouco confusos:
#df.describe()
#Por isso devemos apontar as colunas em que este valores fazem sentido usando ela assim:
#print("\nMedidas Estatísticas Descritivas:")
#print("\nAmplitude Inter Quartis (AIQ): Q0,75-Q0,25")
#print("\nRange: Max - Min")
df[['Idade', 'Altura', 'Peso', 'HorasEstudo']].describe()



Número de linhas: 33
Número de colunas: 10


,Idade,Altura,Peso,HorasEstudo
count,33.000000,33.000000,33.000000,33.000000
mean,21.030303,167.545455,65.181818,7.712121
std,1.590693,8.771726,11.484921,5.455832
min,20.000000,150.000000,46.000000,1.000000
25%,20.000000,161.000000,58.000000,4.000000
50%,21.000000,165.000000,64.000000,7.000000
75%,21.000000,174.000000,68.000000,9.500000
max,28.000000,185.000000,96.000000,28.000000


In [80]:
#Tambem temos esta forma de calcular o anterior, com esta sintaxe podemos ter o "skew" e o kurtosis
num_cols = ['Idade', 'Altura', 'Peso', 'HorasEstudo']

# Estatísticas principais
desc = df[num_cols].agg(['count', 'mean', 'median', 'std', 'min', 'max', 'skew', 'kurtosis']).T

# Quartis (Q1, Q3) + Range
desc['Q1'] = df[num_cols].quantile(0.25)
desc['Q2'] = df[num_cols].quantile(0.50)  # mediana, só para consistência
desc['Q3'] = df[num_cols].quantile(0.75)
desc['Range'] = desc['max'] - desc['min']

# Arredondar e reordenar colunas
desc = desc[['count', 'mean', 'std', 'min', 'Q1', 'Q2', 'Q3', 'max', 'Range', 'skew', 'kurtosis']].round(3)

desc

#skewness() calcula a assimetria da distribuição:
#•	0 → simétrica,
#•	positiva → cauda à direita,
#•	negativa → cauda à esquerda.
#Curtose mede o grau de concentração dos valores em torno da média — distribuições com curtose alta têm caudas mais pesadas (mais valores extremos) e as de curtose baixa são mais achatadas.

,count,mean,std,min,Q1,Q2,Q3,max,Range,skew,kurtosis
Idade,33.0,21.030,1.591,20.0,20.0,21.0,21.0,28.0,8.0,3.022,11.326
Altura,33.0,167.545,8.772,150.0,161.0,165.0,174.0,185.0,35.0,0.216,-0.599
Peso,33.0,65.182,11.485,46.0,58.0,64.0,68.0,96.0,50.0,1.269,1.716
HorasEstudo,33.0,7.712,5.456,1.0,4.0,7.0,9.5,28.0,27.0,1.975,5.392


In [77]:
#Mostra os tipos de variaveis e estrutura do data set
#Dtype = object = string
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Idade          33 non-null     int64   
 1   Sexo           33 non-null     category
 2   Residencia     33 non-null     category
 3   Cohabitação    33 non-null     category
 4   Deslocação     33 non-null     int64   
 5   HorasEstudo    33 non-null     float64 
 6   Altura         33 non-null     int64   
 7   Peso           33 non-null     int64   
 8   Cinema         33 non-null     category
 9   SatisfaçãoLic  33 non-null     category
dtypes: category(5), float64(1), int64(4)
memory usage: 2.4 KB


In [42]:
# 🔹 Variáveis nominais: Sexo, Residência, Cinema
cols_nominais = ['Sexo', 'Residencia', 'Cinema']
df[cols_nominais] = df[cols_nominais].astype('category')

# 🔹 Variável ordinal: Cohabitação (ordem definida manualmente)
cohabit_levels = ['Pais', 'Outros familiares', 'Colegas', 'Sozinho/a']
cohabit_type = CategoricalDtype(categories=cohabit_levels, ordered=True)
df['Cohabitação'] = df['Cohabitação'].astype(cohabit_type)

# 🔹 Variável ordinal: SatisfaçãoLic, se quiseres já definir a ordem
satisf_levels = ['Insatisfeito', 'Pouco Satisfeito', 'Satisfeito', 'Muito Satisfeito']
satisf_type = CategoricalDtype(categories=satisf_levels, ordered=True)
df['SatisfaçãoLic'] = df['SatisfaçãoLic'].astype(satisf_type)

#Para vermos se ficou tudo bem, definimos esta função
def glimpse(df):
    print(f"📊 DataFrame com {df.shape[0]} linhas e {df.shape[1]} colunas\n")
    for col in df.columns:
        dtype = df[col].dtype
        preview = df[col].unique()[:5]  # mostra até 5 valores únicos
        print(f"{col:<15} ({dtype})  →  {preview}")

#print
glimpse(df)

📊 DataFrame com 33 linhas e 10 colunas

Idade           (int64)  →  [20 21 22 24 23]
Sexo            (category)  →  ['Masculino', 'Feminino']
Categories (2, object): ['Feminino', 'Masculino']
Residencia      (category)  →  ['Setúbal', 'Seixal', 'Palmela']
Categories (3, object): ['Palmela', 'Seixal', 'Setúbal']
Cohabitação     (category)  →  ['Pais', 'Colegas', 'Outros familiares', 'Sozinho/a']
Categories (4, object): ['Pais' < 'Outros familiares' < 'Colegas' < 'Sozinho/a']
Deslocação      (int64)  →  [10 30 15 45 25]
HorasEstudo     (float64)  →  [ 3.  8.  6. 12.  4.]
Altura          (int64)  →  [178 170 165 150 161]
Peso            (int64)  →  [83 65 55 46 52]
Cinema          (category)  →  ['Ficção Científica', 'Comédia', 'Aventura', 'Ação', 'Terror']
Categories (5, object): ['Aventura', 'Ação', 'Comédia', 'Ficção Científica', 'Terror']
SatisfaçãoLic   (category)  →  ['Satisfeito', 'Muito Satisfeito', 'Pouco Satisfeito', 'Insatisfeito']
Categories (4, object): ['Insatisfeito' < 'Pou

In [63]:
#Tabela Sexo
freq_abs = df["Sexo"].value_counts() #Frequencias Absolutas
freq_rel = df["Sexo"].value_counts(normalize=True).round(4)*100 #Frequencias Relativas
#Aqui juntamos ambas as columas que calculamos antes em uma tabela
tabelaSexo = pd.DataFrame({
    "Frequência": freq_abs,
    "Frequência Relativa(%)": freq_rel
})

#Tabela Residencia
freq_abs = df["Residencia"].value_counts()
freq_rel = df["Residencia"].value_counts(normalize=True).round(2)
tabelaResidencia = pd.DataFrame({
    "Frequência": freq_abs,
    "Frequência Relativa": freq_rel
})

#Tabela Cohabitação
freq_abs = df["Cohabitação"].value_counts()
freq_rel = df["Cohabitação"].value_counts(normalize=True).round(2)
tabelaCohabitação = pd.DataFrame({
    "Frequência": freq_abs,
    "Frequência Relativa": freq_rel
})

#Tabela Cinema
freq_abs = df["Cinema"].value_counts()
freq_rel = df["Cinema"].value_counts(normalize=True).round(2)
tabelaCinema = pd.DataFrame({
    "Frequência": freq_abs,
    "Frequência Relativa": freq_rel
})

#Tabela SatisfaçãoLic
freq_abs = df["SatisfaçãoLic"].value_counts()
freq_rel = df["SatisfaçãoLic"].value_counts(normalize=True).round(2)
tabelaSatisfaçãoLic = pd.DataFrame({
    "Frequência": freq_abs,
    "Frequência Relativa": freq_rel
})

tabelaSexo


,Frequência,Frequência Relativa(%)
Sexo,,
Feminino,22,66.67
Masculino,11,33.33


In [52]:
tabelaResidencia


,Frequência,Frequência Relativa
Residencia,,
Setúbal,12,0.36
Seixal,11,0.33
Palmela,10,0.30


In [53]:
tabelaCohabitação


,Frequência,Frequência Relativa
Cohabitação,,
Pais,19,0.58
Colegas,11,0.33
Sozinho/a,2,0.06
Outros familiares,1,0.03


In [54]:
tabelaCinema

,Frequência,Frequência Relativa
Cinema,,
Comédia,19,0.58
Aventura,5,0.15
Ação,4,0.12
Ficção Científica,4,0.12
Terror,1,0.03


In [57]:
tabelaSatisfaçãoLic

,Frequência,Frequência Relativa
SatisfaçãoLic,,
Satisfeito,19,0.58
Muito Satisfeito,9,0.27
Pouco Satisfeito,3,0.09
Insatisfeito,2,0.06
